# Beyond Access: A Multi-Source Analysis of Primary Care Shortages, Chronic Disease Burden, and Preventable Hospitalizations Across US Counties

## Overview (add notes from literature)
 
## Problem Statement

Over 92 million Americans live in designated primary care Health Professional Shortage Areas (HPSAs) — a number that grew 21% in 2025 alone — yet the downstream health consequences of these shortages remain poorly quantified at the county level. This project investigates whether primary care shortages are associated with higher chronic disease prevalence and preventable hospitalization rates, and identifies which counties face the compounding burden of shortage, high disease burden, and inadequate safety net coverage.

> **Note:** All findings represent associations at the county level. This analysis cannot 
> establish causal relationships. Confounding factors including poverty, race, and rurality 
> likely mediate observed relationships.

## Data Sources & Methodology 

### Data Sources

**CDC PLACES** (2025) 

CDC PLACES estimates chronic disease and other health-related measures at various geographic levels of the United States using a small area estimation methodology. This dataset contains model-based county estimates. PLACES covers the entire United States—50 states and the District of Columbia. Estimates were provided by the Centers for Disease Control and Prevention (CDC), Division of Population Health, Epidemiology and Surveillance Branch.  This dataset includes estimates for 40 measures: 12 for health outcomes, 7 for preventive services use, 4 for chronic disease-related health risk behaviors, 7 for disabilities, 3 for health status, and 7 for health-related social needs. These estimates can be used to identify emerging health problems and to help develop and carry out effective, targeted public health prevention activities.

**Health Resources & Services Administration (HRSA) HPSA** (Current)

Shortage areas, such as Health Professional Shortage Areas (HPSAs), focus limited resources on communities with the greatest need for health care services. HPSA can be geographic areas, populations, or facilities. These areas have a shortage of primary, dental, or mental health care providers.

**Health Resources & Services Administration (HRSA) MUA/P** (Current)

Medically Underserved Area/Population designations (MUA/P) may be a whole county or a group of contiguous counties, a group of county or civil divisions or a group of urban census tracts in which residents have a shortage of personal health services. Medically Underserved Populations (MUPs) may include groups of persons who face economic, cultural or linguistic barriers to health care. Unlike HPSA which focuses specifically on provider supply, MUA/P measures overall access barriers using a broader Index of Medical Underservice (IMU) score incorporating poverty rate, elderly population, and infant mortality rate.

**Health Resources & Services Administration (HRSA) FQHC** (Current)

Federally Qualified Health Centers (FQHC), Provide primary care to an area or group of people in need, offer a sliding fee scale, provide complete services, have an ongoing quality assurance program, and have a government board of directors.

**County Health Rankings** (2025)

Provides measures of health for counties across the nation. The annual data release is compiled from a variety of national and state data sources. Our collection of over 80 measures of health is organized within topic areas according to the UW Population Health Institute Model of Health. Select measures are used to generate our Health Groups. Focuse is on preventable hospitalization rates, provider density, outcomes. County demographics include population, income, insurance coverage, and race.

**USDA RUCC** (2023) 

The 2023 Rural-Urban Continuum Codes distinguish U.S. metropolitan (metro) counties by the population size of their metro area, and nonmetropolitan (nonmetro) counties by their degree of urbanization and adjacency to a metro area. The division of counties as either metro or nonmetro, based on the 2023 Office of Management and Budget (OMB) delineation of metro areas, is further subdivided into three metro and six nonmetro categories. Each county and census-designated county-equivalent in the United States, including those in outlying territories, is assigned one of these nine codes. The codes allow researchers, policy makers, and others to view county-level data by finer residential groups—beyond metro and nonmetro—when analyzing trends related to population density and metro influence.

**US County Boundaries (GeoJSON)** (Current)

Plotly dataset providing county-level, geographic boundaries for choropleth mapping. Key measures include County FIPS codes, polygon boundaries.

### Methodology
Data from 6 sources was ingested into Snowflake and transformed using a dbt pipeline with staging, intermediate, and mart layers. County-level metrics were joined on FIPS codes to create a unified analytical dataset covering 2,957 US counties.

**Pipeline Architecture:**
- **staging/** → raw source cleaning and standardization (materialized as views)
- **intermediate/** → joins and transformations across sources (materialized as views)
- **marts/** → final analytical tables for analysis and dashboard (materialized as tables)

***Raw Data → Staging → Intermediate → Marts → Analysis/Visualization*** 

- **Staging** — clean and rename raw data
- **Intermediate** — aggregate and prepare for joining
- **Marts** — final business-logic tables (a term from data warehousing that means a focused, analytics-ready subset of data for a specific business domain or use case), one row per county, ready to query
- **Analysis** — SQL queries against marts to answer research questions
- **Visualizations** — Plotly dashboards built on top of analysis queries

**Research questions & SQL analyses** — queries were drafted in VSCode in the `/analyses` folder and executed in Snowflake to answer each research question, including...

- Chronic disease prevalence comparisons between HPSA and non-HPSA counties
- Pearson correlation and quartile analysis of shortage severity vs hospitalization rates
- Triple burden county identification (shortage + high disease + excess hospitalizations)
- FQHC effectiveness controlling for shortage status
- Predictors of preventable hospitalizations (disease burden, PCP ratio, poverty, race, rurality)
- Geographic distribution of shortage + high disease burden counties by state

**Data Export & Visualization** — analytical outputs were exported as CSV to
`data-processed/` for use in the Plotly dashboard:

- `county_health_profile.csv`
- `access_impact_analysis.csv`
- `priority_counties_ranking.csv`
- `regional_patterns.csv`

Dashboard layout was planned in [Figma](https://www.figma.com/design/W51QZYKUW8KN1E2wpoLOE4/Primary-Care-Shortage-Dashboard?node-id=0-1&t=LQVn5JJzD6nr7jeb-1) before development and built using Plotly Dash with a custom dark theme color palette.

> **Note:** Final dataset covers 2,957 of 3,143 US counties. Approximately 186 counties
> were excluded due to missing data in one or more source files during the join process.

### SQL Queries 

#### 1. 

- HPSA-designated counties show consistently higher chronic disease prevalence across all 6 key measures.

- Hypertension showed the largest gap, with HPSA counties averaging 2.29 percentage points higher prevalence than non-HPSA counties (33.81% vs 31.52%).

<img src="../assets/images/SQL_1.png" width="600"/>

#### 2. 

- Pearson's Correlation returns a value between -1 and 1. A Pearson r above 0.3 is generally considered a meaningful positive correlation — r = 0.18. 

- HPSA score alone shows a weak positive correlation with excess preventable stays 
  (r = 0.18), suggesting shortage severity is necessary but not sufficient to explain 
  hospitalization outcomes - motivating a multivariable analysis. 

- HPSA score correlates more strongly with chronic disease prevalence than with hospitalization rates — diabetes (r = 0.39) and hypertension (r = 0.36) show meaningful positive correlations, while obesity shows a weaker relationship (r = 0.15).

- Counties in the highest HPSA severity quartile (avg score 20.6) average +251.8 excess 
  preventable stays above the national average, while the lowest quartile (avg score 11.2) 
  averages -219.5 — a spread of nearly 471 stays per county. 

<img src="../assets/images/SQL_2.png" width="600"/>

#### 3. 

- Counties span AR, LA, MO, KS, OK, ND, AL, GA — crossing Deep South and rural Midwest regions.

- All top-25 counties share maximum vulnerability scores (5/5), dual HPSA+MUA/P designation, and are 92% rural — though this uniformity reflects query filter constraints rather than an emergent pattern.

- Estimated hospital stay counts ranges 114–665, with Clinch County, GA (665) and Marion County, AL (114) at the extremes.

- Grant County, ND records the highest excess preventable stays in the dataset at 5,538 — the largest value despite having a small county population.

<img src="../assets/images/SQL_3.png" width="600"/>

#### 4. 

- Counties with FQHCs average higher excess stays (26.9) than those without (-175.9), a 202.8-point gap that reflects confounding by design — FQHCs are deliberately placed in underserved areas. 

- Within shortage areas, counties with FQHCs average 36.3 excess stays versus -157.2 for shortage counties without FQHCs — a 193.5-point difference. However, the two groups differ in baseline severity: Shortage + FQHC counties carry a higher average HPSA score (16.9) than Shortage, No FQHC counties (14.2).

<img src="../assets/images/SQL_4.png" width="600"/>

#### 5. 

- Very weak correlation — r = 0.03, essentially no linear relationship between IMU score and hospitalization rates. 

- IMU score (which measures medical underservice broadly) shows almost no direct correlation with preventable hospitalizations (r=0.03), while HPSA score (which specifically measures primary care physician shortage severity) shows a weak but meaningful correlation (r=0.18).

- Disease burden level is the strongest predictor of preventable hospitalizations. High burden counties average 481.9 excess stays above the national average, while low burden counties average 368.5 below — a gap of 850 hospitalizations per county.

- Negative and weak correlation - r = -0.16, in the expected direction: counties with more PCPs tend to have fewer excess preventable hospitalizations.

<img src="../assets/images/SQL_5.png" width="600"/>

#### 6. 

- Poverty is the strongest social determinant in this analysis: children in poverty have correlations at r = 0.34 with excess preventable stays.

- Uninsured rate shows a weak positive correlation (r = 0.15), directionally expected but modest — comparable in magnitude to PCP per 100k (r = -0.16) and HPSA score (r = 0.18).

- Racial hospitalization gaps show negligible correlations: Black-white gap (r = -0.02) and Hispanic-white gap (r = 0.05) are effectively zero, suggesting county-level racial disparity in hospitalization rates is not linearly associated with aggregate excess stays in this dataset.

<img src="../assets/images/SQL_6.png" width="600"/>

#### 7. 

- Mississippi is the most affected state — 73.2% of all counties have both shortage and high disease burden, with 645 avg excess stays.

- West Virginia has the highest hospitalization burden — only 33 counties qualify but they average 1,173 excess stays, the highest of any state.

- Louisiana and Mississippi dominate on both count and percentage.

- Texas has the most raw counties (39) but lowest percentage (15.4%) — reflecting its size rather than systematic burden.

<img src="../assets/images/SQL_7.png" width="600"/>


## Key Insights & Findings (add thorough updates)

- **HPSA counties show consistently higher chronic disease burden** across all 12
  conditions analyzed, with hypertension showing the largest gap (+2.29 pp vs non-HPSA).

- **FQHCs alone are insufficient to offset hospitalization burden** they may be treating the sickest counties but can't fully compensate for systemic PCP gaps.

- **The shortage itself may be contributing to worse chronic disease management over time** PCP shortages correlate more strongly with chronic disease than hospitalizations (diabetes r=0.39 vs hospitalization r=0.18), suggesting shortages drive worse disease management over time.

- **Triple burden counties are concentrated in the rural South and Midwest** — top
  priority counties span Georgia, Mississippi, Louisiana, Missouri, Arkansas, Florida, and
  Alaska; 90% are Critical HPSA tier, 9 out of 10 are dual-designated (HPSA + MUA/P).

- **Scale of the problem is significant** — 2,599 HPSA-designated counties, 671 counties
  with high disease burden, and counties in the highest shortage quartile average +259.2
  excess preventable stays above the national average. St. Louis, MO alone has an estimated
  3,379 preventable admissions annually.


**If the 69 highest-priority counties received adequate primary care access, an estimated 194,949 hospitalizations could be prevented annually.** 

> **Note:** Although this figure is an approximation, as preventable stay rates are calculated per 100k Medicare enrollees while total county population was used as the denominator, likely resulting in an overestimate of the true count.

> **Note:** All findings represent associations at the county level. This analysis cannot establish causal relationships between primary care shortages and health outcomes. Confounding factors including poverty, race, and rurality likely mediate observed relationships.

## Policy Recommendations (based on findings + literature)

## Data Visualization / Plotly Dashboard 

![Dashboard](../assets/images/choropleth-maps.png)

![Dashboard](../assets/images/bar-chart-FQHC-effectiveness.png)

![Dashboard](../assets/images/bar-chart-HPSA-shortage-quartile.png)

![Dashboard](../assets/images/bar-chart-HPSA-vs-non-HPSA.png)

![Dashboard](../assets/images/bar-chart-priority-counties-with-PCP-shortage-+-high-disease-burden.png)

![Dashboard](../assets/images/heatmap-triple-burden-counties.png)

![Dashboard](../assets/images/scatter-plot-pcp-density-vs-excess-hospitalizations.png)

![Dashboard](../assets/images/scatter-plot-poverty-vs-excess-hospitalizations.png)

![Dashboard](../assets/images/priority-ranking-table.png)

## Technologies

- **Data Warehouse:** Snowflake
- **Data Transformation:** dbt (Data Build Tool)
- **Analysis:** SQL
- **Visualization:** Plotly
- **Version Control:** Git/GitHub

**GitHub:** [primary-care-shortage-analysis](https://github.com/alinix1/primary-care-shortage-analysis)  